# NB06 · Semana 14 — Serving: gold en PostgreSQL + Metabase (Entrega 4)
**Ingeniería de Datos — LCD · UCA Rosario · 2026** · grupo06 · Clima de Sídney (Open-Meteo)

`scripts/publicar_gold.py` es la última tarea del DAG. Publica `fct_clima_mensual` y `feat_clima_diario`
en el schema `grupo06` de la base `warehouse`.

In [ ]:
import os, sys
from pathlib import Path
# Raíz del proyecto = carpeta padre de notebooks/ (en el servidor: ~/work/dags)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
import entorno
entorno.info()
from sqlalchemy import create_engine, text
from scripts import publicar_gold
engine = create_engine(entorno.pg_url())

## 1. Publicar dos veces y comprobar que no duplica (DELETE + INSERT en una transacción)

In [ ]:
def conteos():
    with engine.connect() as c:
        return {t: c.execute(text(f'select count(*) from "{entorno.PG_SCHEMA}"."{t}"')).scalar() for t in publicar_gold.TABLAS}
publicar_gold.main(); a = conteos(); print("1ª:", a)
publicar_gold.main(); b = conteos(); print("2ª:", b)
assert a == b; print("✔ Publicación repetible")

## 2. Consultar la gold (como lo hará Metabase)

In [ ]:
import pandas as pd
with engine.connect() as c:
    display(pd.read_sql(text(f"""select mes, temperatura_media_c, anomalia_temperatura_c, lluvia_total_mm, dias_con_lluvia
                                  from {entorno.PG_SCHEMA}.fct_clima_mensual order by mes desc limit 6"""), c))

## 3. Tablero en Metabase (https://metabase.nexoacademia.org)
Conexión: base **warehouse**, schema **grupo06**, tablas `fct_clima_mensual` y `feat_clima_diario`.
Las preguntas, títulos y configuración de cada gráfico están en `docs/tablero_metabase.md`
(Metabase no se versiona: ese archivo + las capturas en `docs/capturas/` es lo que viaja con el repo).

> Metabase responde preguntas de **negocio** sobre el warehouse; Grafana se usa para **observabilidad** del pipeline.